In [11]:
import requests
import pandas as pd
import time

# Lista para acumular todos os monstros de todas as páginas
todos_monstros = []

# URL inicial (Página 1)
url_atual = "https://api.open5e.com/v1/monsters/"
pagina = 1

print("Iniciando a extração completa dos dados...")

# O loop continuará rodando enquanto url_atual não for None
while url_atual:
    print(f"Buscando página {pagina}...")
    
    response = requests.get(url_atual)
    
    if response.status_code == 200:
        dados = response.json()
        
        # Adiciona os monstros da página atual à nossa lista global
        todos_monstros.extend(dados['results'])
        
        # Atualiza a URL para a próxima página
        url_atual = dados['next']
        pagina += 1
        
        # Boa prática de engenharia: uma pequena pausa para não sobrecarregar a API pública
        time.sleep(0.3)
    else:
        print(f"\n[ERRO] Falha ao acessar a página {pagina}. Status Code: {response.status_code}")
        break

print(f"\nExtração concluída com sucesso")
print(f"Total de monstros armazenados na memória: {len(todos_monstros)}")

# Transforma a lista completa em um DataFrame consolidado do Pandas
df_completo = pd.DataFrame(todos_monstros)

# Exibe as primeiras linhas para confirmar que funcionou
display(df_completo[['name', 'challenge_rating']].head())

Iniciando a extração completa dos dados...
Buscando página 1...
Buscando página 2...
Buscando página 3...
Buscando página 4...
Buscando página 5...
Buscando página 6...
Buscando página 7...
Buscando página 8...
Buscando página 9...
Buscando página 10...
Buscando página 11...
Buscando página 12...
Buscando página 13...
Buscando página 14...
Buscando página 15...
Buscando página 16...
Buscando página 17...
Buscando página 18...
Buscando página 19...
Buscando página 20...
Buscando página 21...
Buscando página 22...
Buscando página 23...
Buscando página 24...
Buscando página 25...
Buscando página 26...
Buscando página 27...
Buscando página 28...
Buscando página 29...
Buscando página 30...
Buscando página 31...
Buscando página 32...
Buscando página 33...
Buscando página 34...
Buscando página 35...
Buscando página 36...
Buscando página 37...
Buscando página 38...
Buscando página 39...
Buscando página 40...
Buscando página 41...
Buscando página 42...
Buscando página 43...
Buscando página 44..

,name,challenge_rating
0,A-mi-kuk,7
1,Aalpamac,7
2,Aatxe,5
3,Abaasy,8
4,Abbanith Giant,3


In [12]:
# Salvando o arquivo original na nossa pasta de dados brutos
df_completo.to_csv('../data/raw/monstros_brutos.csv', index=False)
print("Dados salvos com sucesso em data/raw/monstros_brutos.csv")

Dados salvos com sucesso em data/raw/monstros_brutos.csv


In [13]:
import re
import ast
import difflib
import pandas as pd

# Mapa otimizado
MAPA_NUMEROS = {
    'once': 1, 'one': 1, '1': 1,
    'twice': 2, 'two': 2, '2': 2,
    'three': 3, '3': 3,
    'four': 4, '4': 4,
    'five': 5, '5': 5,
}

def normalizar_texto(texto):
    #Remove ruído gramatical direto sem afetar marcadores de frequência.
    ruido = r'\b(its|with|using|the|a|an|makes?|attacks?|and|or|use|uses)\b'
    texto = re.sub(ruido, ' ', texto.lower())
    return re.sub(r'\s+', ' ', texto).strip()

def extrair_variantes(nome):
    #Gera formas de singular e plural para o casamento de padrões.
    nome = nome.lower().strip()
    variantes = {nome}
    if nome.endswith('s'):
        variantes.add(nome[:-1])
    else:
        variantes.add(nome + 's')
    if nome.endswith('es'):
        variantes.add(nome[:-2])
    return variantes

def parse_multiattack_local(desc_multiattack, danos_individuais):
    #Decifra o combo do Multiattack usando o pipeline otimizado em três camadas: Regex Direto -> Regex Reverso -> Proximidade Fuzzy.
    
    texto_norm = normalizar_texto(desc_multiattack)
    dano_total = 0
    
    for nome_ataque, dano_base in danos_individuais.items():
        variantes = extrair_variantes(nome_ataque)
        quantidade_encontrada = 0
        
        for variante in variantes:
            # Padrão 1 Otimizado: Número colado ANTES do nome (ex: "two claw")
            padrao_antes = rf'\b({"|".join(MAPA_NUMEROS.keys())})\s+{re.escape(variante)}\b'
            match = re.search(padrao_antes, texto_norm)
            if match:
                quantidade_encontrada = MAPA_NUMEROS.get(match.group(1), 1)
                break
                
            # Padrão 2: Número colado DEPOIS do nome (ex: "claw two")
            padrao_depois = rf'\b{re.escape(variante)}\s+({"|".join(MAPA_NUMEROS.keys())})\b'
            match = re.search(padrao_depois, texto_norm)
            if match:
                quantidade_encontrada = MAPA_NUMEROS.get(match.group(1), 1)
                break
        
        # Padrão 3: Fallback por similaridade (Fuzzy) nas proximidades do texto
        if quantidade_encontrada == 0:
            palavras = texto_norm.split()
            for i, palavra in enumerate(palavras):
                if difflib.get_close_matches(palavra, variantes, n=1, cutoff=0.82):
                    vizinhos = palavras[max(0, i-3):i] + palavras[i+1:i+4]
                    for v in vizinhos:
                        if v in MAPA_NUMEROS:
                            quantidade_encontrada = MAPA_NUMEROS[v]
                            break
                    if quantidade_encontrada == 0:
                        quantidade_encontrada = 1
                    break
                    
        dano_total += quantidade_encontrada * dano_base
        
    return dano_total

def descobrir_multiplicador_cooldown(texto_acao, nome_acao):
    texto_analise = f"{nome_acao} {texto_acao}".lower()
    
    usos_dia = re.search(r'(\d+)/day', texto_analise)
    if usos_dia:
        return min(int(usos_dia.group(1)), 3) / 3
        
    recharge_intervalo = re.search(r'recharge\s*(\d+)–(\d+)', texto_analise)
    if recharge_intervalo:
        p = ((int(recharge_intervalo.group(2)) - int(recharge_intervalo.group(1))) + 1) / 6
        return (1 + 2*p) / 3
        
    recharge_unico = re.search(r'recharge\s*(\d+)', texto_analise)
    if recharge_unico and '6' in recharge_unico.group(1):
        return (1 + 2*(1/6)) / 3
            
    return 1.0

def calcular_dpr_single_target_v8(row):
    acoes_normais = row['actions']
    acoes_lendarias = row['legendary_actions']
    
    if isinstance(acoes_normais, str):
        try: acoes_normais = ast.literal_eval(acoes_normais)
        except: acoes_normais = []
    if not isinstance(acoes_normais, list): acoes_normais = []
        
    danos_individuais = {}
    maior_ataque_isolado_com_cooldown = 0
    desc_multiattack = ""
    
    for acao in acoes_normais:
        name = acao.get('name', '').lower().strip()
        desc = acao.get('desc', '')
        
        if name == 'multiattack':
            desc_multiattack = desc
            continue
            
        padroes = re.findall(r'\((\d+)d(\d+)\s*([+-]\s*\d+)?\)', desc)
        dano_da_acao = 0
        for num_dados, faces_dado, bonus in padroes:
            bonus_int = int(bonus.replace(' ', '')) if bonus else 0
            dano_da_acao += ((int(faces_dado) / 2) + 0.5) * int(num_dados) + bonus_int
            
        danos_individuais[name] = dano_da_acao
        
        mult_cooldown = descobrir_multiplicador_cooldown(desc, name)
        dano_isolado = dano_da_acao * mult_cooldown
        if dano_isolado > maior_ataque_isolado_com_cooldown:
            maior_ataque_isolado_com_cooldown = dano_isolado

    dano_multiattack_total = 0
    if desc_multiattack and danos_individuais:
        dano_multiattack_total = parse_multiattack_local(desc_multiattack, danos_individuais)
                
    dano_base_turno = max(dano_multiattack_total, maior_ataque_isolado_com_cooldown)
    
    dano_lendario_total = 0
    if isinstance(acoes_lendarias, str):
        try: acoes_lendarias = ast.literal_eval(acoes_lendarias)
        except: acoes_lendarias = []
        
    if isinstance(acoes_lendarias, list):
        maior_dano_lendario = 0
        for acao in acoes_lendarias:
            desc = acao.get('desc', '')
            name = acao.get('name', '')
            
            custo = 1
            match_custo = re.search(r'costs\s*(\d+)\s*actions', f"{name} {desc}".lower())
            if match_custo:
                custo = int(match_custo.group(1))
            
            padroes = re.findall(r'\((\d+)d(\d+)\s*([+-]\s*\d+)?\)', desc)
            dano_lendario_acao = 0
            for num_dados, faces_dado, bonus in padroes:
                bonus_int = int(bonus.replace(' ', '')) if bonus else 0
                dano_lendario_acao += ((int(faces_dado) / 2) + 0.5) * int(num_dados) + bonus_int
                
            dano_ajustado = (dano_lendario_acao * descobrir_multiplicador_cooldown(desc, name)) / custo
            if dano_ajustado > maior_dano_lendario:
                maior_dano_lendario = dano_ajustado
                
        dano_lendario_total = maior_dano_lendario * 3
        
    return round(dano_base_turno + dano_lendario_total, 2)

# Aplica apenas nos monstros oficiais, poupando processamento e isolando a base limpa
df_oficial = df_completo[df_completo['document__slug'] == 'wotc-srd'].copy()
df_oficial['dpr_maximo'] = df_oficial.apply(calcular_dpr_single_target_v8, axis=1)

display(df_oficial.sort_values(by='dpr_maximo', ascending=False)[['name', 'cr', 'dpr_maximo']].head(10))

,name,cr,dpr_maximo
2773,Tarrasque,30.0,204.0
2613,Solar,21.0,140.0
2203,Pit Fiend,20.0,120.5
159,Ancient Red Dragon,24.0,116.5
139,Ancient Blue Dragon,23.0,112.0
144,Ancient Bronze Dragon,22.0,112.0
153,Ancient Green Dragon,22.0,99.5
151,Ancient Gold Dragon,24.0,97.0
170,Ancient White Dragon,20.0,94.5
165,Ancient Silver Dragon,23.0,93.0


In [14]:
import ast


def extrair_save_dc_max(row):
    textos = []
    for campo in ['actions', 'special_abilities', 'legendary_actions']:
        val = row.get(campo, [])
        if isinstance(val, str):
            try: val = ast.literal_eval(val)
            except: val = []
        if isinstance(val, list):
            for item in val:
                textos.append(item.get('desc', ''))
    spells_texto = row.get('spells', '')
    if isinstance(spells_texto, str):
        textos.append(spells_texto)
    dcs = re.findall(r'DC\s*(\d+)', ' '.join(textos).upper())
    return max([int(dc) for dc in dcs], default=0)

def contar_tipos_dano(texto):
    if not isinstance(texto, str) or texto.strip() == '':
        return 0
    tipos_oficiais = [
        'acid', 'bludgeoning', 'cold', 'fire', 'force', 'lightning',
        'necrotic', 'piercing', 'poison', 'psychic', 'radiant', 'slashing', 'thunder'
    ]
    return sum(1 for tipo in tipos_oficiais if tipo in texto.lower())

def contar_habilidades_especiais(val):
    if isinstance(val, str):
        try: val = ast.literal_eval(val)
        except: return 0
    if isinstance(val, list):
        return len(val)
    return 0

df_oficial['save_dc_max'] = df_oficial.apply(extrair_save_dc_max, axis=1)
df_oficial['qtd_imunidades'] = df_oficial['damage_immunities'].apply(contar_tipos_dano)
df_oficial['qtd_resistencias'] = df_oficial['damage_resistances'].apply(contar_tipos_dano)
df_oficial['qtd_habilidades_especiais'] = df_oficial['special_abilities'].apply(contar_habilidades_especiais)

colunas_finais = [
    'name', 'cr', 'armor_class', 'hit_points', 'dpr_maximo',
    'strength', 'dexterity', 'constitution', 'intelligence', 'wisdom', 'charisma',
    'save_dc_max', 'qtd_imunidades', 'qtd_resistencias', 'qtd_habilidades_especiais'
]

df_ml = df_oficial[colunas_finais].dropna(subset=['cr']).copy()
df_ml.to_csv('../data/processed/monstros_srd_ml_v2.csv', index=False)

print(f"Dataset V2 gerado com {df_ml.shape[0]} monstros oficiais e {len(colunas_finais)-2} features")
display(df_ml[['name', 'cr', 'save_dc_max', 'qtd_imunidades', 'qtd_habilidades_especiais']].sort_values('save_dc_max', ascending=False).head(10))

Dataset V2 gerado com 322 monstros oficiais e 13 features


,name,cr,save_dc_max,qtd_imunidades,qtd_habilidades_especiais
1726,Kraken,23.0,25,4,3
151,Ancient Gold Dragon,24.0,25,1,2
2613,Solar,21.0,25,2,4
159,Ancient Red Dragon,24.0,25,1,1
165,Ancient Silver Dragon,23.0,25,1,1
139,Ancient Blue Dragon,23.0,24,1,1
144,Ancient Bronze Dragon,22.0,24,1,2
137,Ancient Black Dragon,21.0,23,1,2
153,Ancient Green Dragon,22.0,23,1,2
146,Ancient Copper Dragon,21.0,23,1,1
